## RNN: Thorcino ways

The aim of this notebook is to show how to train a classic RNN using this framework


In [1]:
import numpy as np

N = 10
T = 5

def create_sequence(len: int, start: float, stop: float) -> np.ndarray:
    return np.linspace(start, stop, len)

def create_random_sequences(n: int, len:int, min_start: int=10, seed: float=777) -> np.ndarray:
    rng = np.random.default_rng(seed)
    sequences = []
    for _ in range(n):
        start = rng.integers(1, min_start, size=1)[0]
        stop = start * 3
        seq = create_sequence(len+1, start, stop)

        sequences.append(seq)

    return np.stack(sequences)

Train = create_random_sequences(N, T)
Test = create_random_sequences(N, T, min_start=7)


X_tr, Y_tr = Train[:, :-1], Train[:, 1:]
X_te, Y_te = Test[:, :-1], Test[:, 1:]

print(X_tr)
print(Y_tr)

[[ 9.  12.6 16.2 19.8 23.4]
 [ 6.   8.4 10.8 13.2 15.6]
 [ 4.   5.6  7.2  8.8 10.4]
 [ 4.   5.6  7.2  8.8 10.4]
 [ 1.   1.4  1.8  2.2  2.6]
 [ 6.   8.4 10.8 13.2 15.6]
 [ 5.   7.   9.  11.  13. ]
 [ 9.  12.6 16.2 19.8 23.4]
 [ 3.   4.2  5.4  6.6  7.8]
 [ 2.   2.8  3.6  4.4  5.2]]
[[12.6 16.2 19.8 23.4 27. ]
 [ 8.4 10.8 13.2 15.6 18. ]
 [ 5.6  7.2  8.8 10.4 12. ]
 [ 5.6  7.2  8.8 10.4 12. ]
 [ 1.4  1.8  2.2  2.6  3. ]
 [ 8.4 10.8 13.2 15.6 18. ]
 [ 7.   9.  11.  13.  15. ]
 [12.6 16.2 19.8 23.4 27. ]
 [ 4.2  5.4  6.6  7.8  9. ]
 [ 2.8  3.6  4.4  5.2  6. ]]


In [2]:
"""Model architecture"""

from thorcino.activations import Identity
from thorcino.layers import Sequential, RNN
from thorcino.losses import MSELoss
from thorcino.optimizer import SGD
from thorcino.training.schedulers import CosineSchedule
from thorcino.training.trainer import Trainer 

EPOCHS = 100
IN_FEATURE = 1
OUT_FEATURE = 1
HIDDEN_UNITS = 2
MAX_LR, MIN_LR = 1e-2, 1e-5

model = Sequential(
    RNN(IN_FEATURE, OUT_FEATURE, HIDDEN_UNITS, activation=Identity()),
    Identity()
)
model.save_graph("./images/arch.png", True)
model.save_graph("./images/forward.png", False, True)
model.save_graph("./images/backward.png", False, backward=True)

loss = MSELoss()
optimizer = SGD(model.parameters, MAX_LR)
scheduler = CosineSchedule(MAX_LR, MIN_LR, EPOCHS)
trainer = Trainer(model, loss, optimizer, scheduler)

In [3]:
"""Preprocessing"""

from thorcino.dataset.dataset import DataLoader, TensorDataset
from thorcino.tensor import Tensor

X_tr, Y_tr = Tensor(X_tr), Tensor(Y_tr)
dataset_tr = TensorDataset(X_tr, Y_tr)
loader_tr = DataLoader(dataset_tr, N)

X_te, Y_te = Tensor(X_te), Tensor(Y_te)
dataset_te = TensorDataset(X_te, Y_te)
loader_te = DataLoader(dataset_te, N)

In [4]:
"""Training"""

EVAL_STEP = EPOCHS / 10

for e in range(EPOCHS):
    trainer.train_epoch(dataset_tr)

    if e%EVAL_STEP == 0:
        trainer.eval(dataset_te)

Tensor([ 9.  12.6 16.2 19.8 23.4]) Tensor([12.6 16.2 19.8 23.4 27. ]) Tensor([[[1.6073601]]

 [[2.8204577]]

 [[4.4029903]]

 [[6.181562 ]]

 [[8.170962 ]]])


ValueError: non-broadcastable output operand with shape (5,1,1) doesn't match the broadcast shape (5,1,5)